# Solutions — Array, Map, Struct & JSON Extras (Medium 21)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_customers    = spark.table("samples.bakehouse.sales_customers")
df_bookings     = spark.table("samples.wanderbricks.bookings")

## Solution 1 — explode_outer vs posexplode_outer

In [ ]:
df_tags = spark.createDataFrame([(1,["a","b","c"]),(2,None),(3,["x"])], ["id","tags"])
result_1a = df_tags.select("id", F.explode("tags").alias("tag"))
result_1b = df_tags.select("id", F.explode_outer("tags").alias("tag"))
result_1c = df_tags.select("id", F.posexplode_outer("tags").alias("pos","tag"))

## Solution 2 — array_remove, array_position, array_repeat

In [ ]:
df_nums = spark.createDataFrame([(1,[1,2,3,2,1]),(2,[5,5,5])], ["id","nums"])
result_2 = (
    df_nums
    .withColumn("removed_2", F.array_remove("nums", 2))
    .withColumn("pos_of_5",  F.array_position("nums", 5))
    .withColumn("repeated",  F.array_repeat(F.lit(0), 3))
)

## Solution 3 — arrays_overlap and arrays_zip

In [ ]:
df_ab = spark.createDataFrame([(1,[1,2,3],[2,4]),(2,[5,6],[7,8])], ["id","a","b"])
result_3 = (
    df_ab
    .withColumn("has_overlap", F.arrays_overlap("a","b"))
    .withColumn("zipped",      F.arrays_zip("a","b"))
)

## Solution 4 — map_entries, map_filter, map_concat

In [ ]:
result_4 = (
    df_transactions
    .withColumn("product_price_map",
        F.create_map(F.col("product"), F.col("totalPrice")))
    .withColumn("map_entries_col", F.map_entries("product_price_map"))
    .withColumn("filtered_map",
        F.map_filter("product_price_map", lambda k, v: v > 10))
    .withColumn("concat_map",
        F.map_concat(
            "product_price_map",
            F.create_map(F.lit("paymentMethod"), F.col("totalPrice"))
        ))
    .select("transactionID","product_price_map","map_entries_col","filtered_map","concat_map")
    .limit(100)
)

## Solution 5 — str_to_map: Parse String to Map

In [ ]:
df_kv = spark.createDataFrame([("a=1,b=2",),("c=3,d=4",)], ["kv_string"])
result_5 = (
    df_kv
    .withColumn("kv_map", F.str_to_map("kv_string", F.lit(","), F.lit("=")))
    .withColumn("val_a",  F.col("kv_map")["a"])
)

## Solution 6 — named_struct and Field Access

In [ ]:
result_6 = (
    df_transactions
    .withColumn("transaction_summary",
        F.struct(F.col("transactionID").alias("id"), F.col("totalPrice").alias("price")))
    .withColumn("summary_id",    F.col("transaction_summary.id"))
    .withColumn("summary_price", F.col("transaction_summary.price"))
    .select("transactionID","totalPrice","transaction_summary","summary_id","summary_price")
    .limit(100)
)

## Solution 7 — json_tuple and get_json_object

In [ ]:
df_json = spark.createDataFrame(
    [('{"name":"Alice","score":95,"city":"NYC"}',),
     ('{"name":"Bob","score":87,"city":"LA"}',)], ["json_str"])
result_7 = (
    df_json
    .select(
        "json_str",
        F.json_tuple("json_str","name","score","city").alias("c0","c1","c2"),
        F.get_json_object("json_str","$.name").alias("name_gjo")
    )
)

## Solution 8 — sort_array, array_distinct, array_union on Real Data

In [ ]:
result_8 = (
    df_transactions
    .groupBy("franchiseID")
    .agg(F.collect_list("product").alias("all_products"))
    .withColumn("sorted_products",      F.sort_array("all_products"))
    .withColumn("unique_products",      F.array_distinct("all_products"))
    .withColumn("product_count_before", F.size("all_products"))
    .withColumn("product_count_after",  F.size("unique_products"))
    .select("franchiseID","all_products","sorted_products","unique_products",
            "product_count_before","product_count_after")
    .orderBy("franchiseID")
)

# array_union demo: take two franchises and union their unique product lists
_franchise_pairs = (
    result_8.alias("a")
    .crossJoin(result_8.alias("b"))
    .filter(F.col("a.franchiseID") < F.col("b.franchiseID"))
    .select(
        F.col("a.franchiseID").alias("franchiseID_a"),
        F.col("b.franchiseID").alias("franchiseID_b"),
        F.array_union(F.col("a.unique_products"), F.col("b.unique_products")).alias("union_products")
    )
)
result_8_union = _franchise_pairs.limit(5)